# Onion Disease & Pest Classification — OnionGuard ML Pipeline
**Assigned to**: Nathaniel Adika  
**Dataset**: TOM2024 (Onion diseases + pests)  
**Platform**: Kaggle Notebook (GPU)  
**Model**: MobileNetV3-Large (ImageNet pretrained)  
**Target**: >90% accuracy, <4MB TFLite quantized model  
**Classes**: 6 (Alternaria, Bulb Blight, Caterpillar, Fusarium, Healthy, Virosis)

### Pipeline Overview
1. **EDA** — Class Distribution, Image Stats, Pixel Intensity, Sample Grids
2. **Data Preparation** — Train/Val/Test splits, class weights for imbalance
3. **Model Training** — 3-stage progressive fine-tuning (AdamW + Cosine LR)
4. **Evaluation** — Confusion matrix, classification report, misclassified samples
5. **Grad-CAM** — Interpretability / XAI visualizations
6. **TFLite Export** — INT8 quantization for mobile deployment

### Techniques Used
| Component | Technique |
|-----------|-----------|
| Model | MobileNetV3-Large (ImageNet pretrained) |
| Input | 224×224 |
| Optimizer | AdamW (lr=1e-4, weight_decay=0.01) |
| LR Schedule | Cosine annealing + linear warmup |
| Augmentation | RandomFlip/Rotation/Zoom/Brightness/Contrast |
| Regularization | Dropout(0.3) + Class weights |
| Fine-tuning | 3-stage progressive unfreezing |
| Quantization | INT8 Post-Training Quantization |
| XAI | Grad-CAM on last conv layer |

---
> **Checkpointing**: Every major step saves results to `/kaggle/working/` so you can download them from the Output tab after the run.


In [ ]:
# ── 0.0 Clean up (only deletes combined_dataset, keeps checkpoints) ───────
!rm -rf /kaggle/working/combined_dataset
print("Cleaned up combined_dataset. Checkpoints preserved.")


In [ ]:
# ── 0.1 Install deps & setup paths (Kaggle) ───────────────────────────────────────
import os, pickle

!pip install -q opencv-python-headless

# ── Kaggle dataset path (added via "+ Add Data" in notebook sidebar) ──────
KAGGLE_DATASET = "/kaggle/input/datasets/souravparija3025/tom2024-dataset/TOM2024_dataset"

if not os.path.exists(KAGGLE_DATASET):
    raise FileNotFoundError(
        f"{KAGGLE_DATASET} not found. "
        "Make sure you added the tom2024-dataset via + Add Data in the right panel."
    )
print(f"Dataset found at: {KAGGLE_DATASET}")

import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from tensorflow.keras.applications import MobileNetV3Large
from sklearn.metrics import classification_report, confusion_matrix
import cv2

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

sns.set_style('whitegrid')
%matplotlib inline

# ── BASE path (Kaggle — no extraction needed, dataset is pre-mounted) ─────
BASE = Path(KAGGLE_DATASET)
print(f"BASE: {BASE}")

# ── Outputs saved to /kaggle/working/ (downloadable from Output tab) ───────
OUT_DIR  = Path("/kaggle/working/caterpillar_output")
CKPT_DIR = OUT_DIR / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Checkpoint helpers ───────────────────────────────────────────────────────────
def save_checkpoint(name, data):
    with open(CKPT_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(data, f)
    print(f"  [CHECKPOINT SAVED] {name}")

def load_checkpoint(name):
    path = CKPT_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  [CHECKPOINT LOADED] {name}")
        return data
    return None

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if tf.config.list_physical_devices('GPU'):
    print(f"GPU: {tf.config.list_physical_devices('GPU')[0]}")
print(f"Output dir : {OUT_DIR}")
print(f"Checkpoints: {CKPT_DIR}")
print("All imports OK")


## Part 1 — EDA & Class Distribution Audit

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 6
CLASS_NAMES = [
    "Alternaria",
    "Bulb_Blight",
    "Caterpillar",
    "Fusarium",
    "Healthy",
    "Virosis",
]
EPOCHS_PHASE1 = 20
EPOCHS_PHASE2 = 25
EPOCHS_PHASE3 = 15

# ── Dataset paths (all onion classes) ─────────────────────────────────
RAW_PATHS = {
    "Alternaria":   BASE / "onion_diseases" / "Alternaria_D",
    "Bulb_Blight":  BASE / "onion_diseases" / "Bulb_blight-D",
    "Caterpillar":  BASE / "onion_pests" / "Caterpillar-P",
    "Fusarium":     BASE / "onion_diseases" / "Fusarium-D",
    "Healthy":      BASE / "onion_diseases" / "Healthy_leaf",
    "Virosis":      BASE / "onion_diseases" / "Virosis-D",
}

# These are not used for multi-class (no augmented folder), but keep for EDA compat
AUG_TRAIN = RAW_PATHS.copy()
AUG_TEST = RAW_PATHS.copy()

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

def list_images(folder: Path) -> list:
    if not folder.exists():
        print(f"  WARNING: {folder} does not exist!")
        return []
    return [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]

# Verify paths and count images
print("Path verification:")
total_images = 0
class_counts = {}
for name, p in RAW_PATHS.items():
    count = len(list_images(p)) if p.exists() else 0
    class_counts[name] = count
    total_images += count
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {name}: {p} ({count} images)")
print(f"\nTotal: {total_images} images across {NUM_CLASSES} classes")

# ── Compute class weights to handle imbalance ──────────────────────────
from sklearn.utils.class_weight import compute_class_weight
total = sum(class_counts.values())
n_classes = len(class_counts)
class_weight_dict = {}
for i, name in enumerate(CLASS_NAMES):
    count = class_counts[name]
    weight = total / (n_classes * count)
    # Cap weights to max 5.0 to prevent tiny classes from destabilizing training
    class_weight_dict[i] = min(weight, 5.0)
print(f"\nClass weights (to handle imbalance):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {class_weight_dict[i]:.2f} ({class_counts[name]} images)")


## 2. Image Counts & Class Distribution

In [ ]:
rows = []
for label in CLASS_NAMES:
    raw = list_images(RAW_PATHS[label])
    rows.append({
        "Class": label,
        "Images": len(raw),
    })

df_counts = pd.DataFrame(rows)
df_counts


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors_map = ["#e74c3c", "#e67e22", "#f1c40f", "#3498db", "#27ae60", "#9b59b6"]
bars = ax.bar(df_counts["Class"], df_counts["Images"],
              color=colors_map, edgecolor="black")
ax.set_title("Onion Disease & Pest — Image Counts", fontsize=14, fontweight="bold")
ax.set_ylabel("Image Count")
ax.set_xlabel("Class")
plt.xticks(rotation=30, ha="right")

for bar, v in zip(bars, df_counts["Images"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 10, str(v),
            ha="center", fontweight="bold")

plt.tight_layout()
fig.savefig(OUT_DIR / "01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
save_checkpoint("eda_class_dist", df_counts.to_dict())


## 3. Image Dimensions & File Size Analysis

In [ ]:
def get_image_stats(img_paths: list, sample_n: int = 300) -> dict:
    """Compute width, height, file size, and channel stats from a sample."""
    sample = random.sample(img_paths, min(sample_n, len(img_paths)))
    widths, heights, file_sizes, channels = [], [], [], []
    corrupt_count = 0

    for p in sample:
        try:
            file_sizes.append(os.path.getsize(p) / 1024)  # KB
            with Image.open(p) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                channels.append(len(img.getbands()))
        except Exception:
            corrupt_count += 1

    return {
        "widths": np.array(widths),
        "heights": np.array(heights),
        "file_sizes_kb": np.array(file_sizes),
        "channels": Counter(channels),
        "corrupt": corrupt_count,
        "sampled": len(sample),
    }

all_stats = {}
for label, path in RAW_PATHS.items():
    imgs = list_images(path)
    stats = get_image_stats(imgs)
    all_stats[label] = stats

    print(f"\n--- {label} (sampled {stats['sampled']} images) ---")
    print(f"  Width  — min: {stats['widths'].min()}, max: {stats['widths'].max()}, "
          f"mean: {stats['widths'].mean():.0f}, median: {np.median(stats['widths']):.0f}")
    print(f"  Height — min: {stats['heights'].min()}, max: {stats['heights'].max()}, "
          f"mean: {stats['heights'].mean():.0f}, median: {np.median(stats['heights']):.0f}")
    print(f"  File size (KB) — min: {stats['file_sizes_kb'].min():.1f}, "
          f"max: {stats['file_sizes_kb'].max():.1f}, mean: {stats['file_sizes_kb'].mean():.1f}")
    print(f"  Channels: {dict(stats['channels'])}")
    if stats['corrupt'] > 0:
        print(f"  WARNING — Corrupt/unreadable: {stats['corrupt']}")

In [ ]:
# Dimension scatter plot
colors = {name: c for name, c in zip(CLASS_NAMES,
          ["#e74c3c", "#e67e22", "#f1c40f", "#3498db", "#27ae60", "#9b59b6"])}

n_classes = len(all_stats)
cols = min(3, n_classes)
rows = (n_classes + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten()

for idx, (label, stats) in enumerate(all_stats.items()):
    axes[idx].scatter(stats["widths"], stats["heights"],
                      alpha=0.5, color=colors.get(label, "gray"), edgecolors="black", linewidths=0.3)
    axes[idx].set_title(f"{label} — Width vs Height", fontweight="bold")
    axes[idx].set_xlabel("Width (px)")
    axes[idx].set_ylabel("Height (px)")
    axes[idx].axhline(y=224, color="gray", linestyle="--", alpha=0.5, label="224px")
    axes[idx].axvline(x=224, color="gray", linestyle="--", alpha=0.5)
    axes[idx].legend(fontsize=8)

# Hide unused axes
for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(OUT_DIR / "02_dimension_scatter.png", dpi=150)
plt.show()


In [ ]:
# File size distribution
fig, ax = plt.subplots(figsize=(12, 5))
for label, stats in all_stats.items():
    ax.hist(stats["file_sizes_kb"], bins=30, alpha=0.5,
            label=label, color=colors.get(label, "gray"), edgecolor="black")
ax.set_xlabel("File Size (KB)")
ax.set_ylabel("Frequency")
ax.set_title("File Size Distribution (Raw Images)", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUT_DIR / "03_filesize_distribution.png", dpi=150)
plt.show()


## 4. Sample Image Grid

In [ ]:
for label, path in RAW_PATHS.items():
    imgs = list_images(path)
    n_show = min(12, len(imgs))
    if n_show == 0:
        print(f"Skipping {label} — no images")
        continue
    sample = random.sample(imgs, n_show)

    n_cols = min(6, n_show)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    fig.suptitle(f"{label} — Sample Images ({len(imgs)} total)",
                 fontsize=14, fontweight="bold")

    for idx, ax in enumerate(axes):
        if idx < len(sample):
            img = Image.open(sample[idx]).convert("RGB")
            ax.imshow(np.array(img))
            ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(OUT_DIR / f"04_samples_{label}.png", dpi=150)
    plt.show()


## 5. Pixel Intensity (RGB Channel Means)

In [ ]:
n_classes = len(RAW_PATHS)
cols = min(3, n_classes)
rows = (n_classes + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes = np.array(axes).flatten()

for idx, (label, path) in enumerate(RAW_PATHS.items()):
    imgs = list_images(path)
    sample = random.sample(imgs, min(100, len(imgs)))
    r_means, g_means, b_means = [], [], []

    for p in sample:
        try:
            img = np.array(Image.open(p).convert("RGB"))
            r_means.append(img[:, :, 0].mean())
            g_means.append(img[:, :, 1].mean())
            b_means.append(img[:, :, 2].mean())
        except Exception:
            pass

    axes[idx].hist(r_means, bins=20, alpha=0.6, color="red", label="R")
    axes[idx].hist(g_means, bins=20, alpha=0.6, color="green", label="G")
    axes[idx].hist(b_means, bins=20, alpha=0.6, color="blue", label="B")
    axes[idx].set_title(f"{label}", fontweight="bold", fontsize=10)
    axes[idx].set_xlabel("Mean Pixel Value")
    axes[idx].legend(fontsize=8)

    print(f"--- {label} ---")
    print(f"  R: mean={np.mean(r_means):.1f}, G: mean={np.mean(g_means):.1f}, B: mean={np.mean(b_means):.1f}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("RGB Channel Means per Class", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(OUT_DIR / "05_pixel_intensity.png", dpi=150)
plt.show()


## 6. Aspect Ratio Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label, stats in all_stats.items():
    ratios = stats["widths"] / stats["heights"]
    ax.hist(ratios, bins=30, alpha=0.5, label=label,
            color=colors.get(label, "gray"), edgecolor="black")
    print(f"  {label}: mean ratio={ratios.mean():.2f}, std={ratios.std():.2f}")

ax.set_xlabel("Aspect Ratio (W/H)")
ax.set_ylabel("Frequency")
ax.set_title("Aspect Ratio Distribution", fontweight="bold")
ax.axvline(x=1.0, color="gray", linestyle="--", label="Square (1:1)")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUT_DIR / "06_aspect_ratio.png", dpi=150)
plt.show()


## 7. Augmented vs Raw — Comparison

In [ ]:
# Show sample images from each class
for label in CLASS_NAMES:
    raw_imgs = list_images(RAW_PATHS[label])
    if len(raw_imgs) == 0:
        print(f"Skipping {label} — no images")
        continue

    n_show = min(6, len(raw_imgs))
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f"{label} — Sample Images ({len(raw_imgs)} total)",
                 fontsize=12, fontweight="bold")

    sample = random.sample(raw_imgs, n_show)
    for i, ax in enumerate(axes):
        img = Image.open(sample[i]).convert("RGB")
        ax.imshow(np.array(img))
        ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(OUT_DIR / f"07_samples_{label}.png", dpi=150)
    plt.show()


## 1.8 EDA Summary

In [ ]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)
print(f"\nDataset: TOM2024 — Onion Disease & Pest Classification")
print(f"Classes: {NUM_CLASSES}")
print()
for i, row in df_counts.iterrows():
    print(f"  {row['Class']:20s}: {row['Images']:4d} images")
print(f"\n  {'TOTAL':20s}: {df_counts['Images'].sum():4d} images")
print(f"\nSmallest class: {df_counts.loc[df_counts['Images'].idxmin(), 'Class']} ({df_counts['Images'].min()} images)")
print(f"Largest class:  {df_counts.loc[df_counts['Images'].idxmax(), 'Class']} ({df_counts['Images'].max()} images)")
print(f"Imbalance ratio: {df_counts['Images'].max() / df_counts['Images'].min():.1f}x")
print("\nClass weights will be used during training to handle imbalance.")

save_checkpoint("eda_counts", df_counts.to_dict())


---
## Part 2 — Data Preparation
Load Category B augmented data, split train into train/val (85/15), build tf.data pipelines with CutMix and Mixup augmentation.

In [ ]:
# ── 2.1 Build tf.data pipelines from all onion classes ─────────────────
import shutil

COMBINED_DIR = Path("/kaggle/working/combined_dataset")

# Remove old combined dataset to start fresh
if COMBINED_DIR.exists():
    shutil.rmtree(COMBINED_DIR)

for class_name, src_path in RAW_PATHS.items():
    dst = COMBINED_DIR / class_name
    dst.mkdir(parents=True, exist_ok=True)
    imgs = list_images(src_path)
    for img_file in imgs:
        os.symlink(str(img_file), str(dst / img_file.name))
    print(f"  {class_name}: {len(imgs)} images")

print(f"Created combined dataset at {COMBINED_DIR}")

# 80/20 split: training vs held-out
train_ds = tf.keras.utils.image_dataset_from_directory(
    COMBINED_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training",
)

held_out_ds = tf.keras.utils.image_dataset_from_directory(
    COMBINED_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
)

# Unbatch and shuffle to ensure all classes in val and test
held_out_unbatched = held_out_ds.unbatch().shuffle(2000, seed=42)
total_held = sum(1 for _ in held_out_unbatched)
val_count = total_held // 2
print(f"Total held-out images: {total_held}, Val: {val_count}, Test: {total_held - val_count}")

# Re-shuffle with same seed for reproducibility
held_out_unbatched = held_out_ds.unbatch().shuffle(2000, seed=42)
val_ds = held_out_unbatched.take(val_count).batch(BATCH_SIZE)
test_ds = held_out_unbatched.skip(val_count).batch(BATCH_SIZE)

train_batches = tf.data.experimental.cardinality(train_ds).numpy()
val_batches = sum(1 for _ in val_ds)
test_batches = sum(1 for _ in test_ds)

print(f"\nTrain batches: {train_batches}")
print(f"Val batches:   {val_batches}")
print(f"Test batches:  {test_batches}")

# Verify all classes in each split
for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
    all_labels = []
    for _, lbls in ds:
        all_labels.extend(np.argmax(lbls.numpy(), axis=1))
    unique, counts = np.unique(all_labels, return_counts=True)
    print(f"\n{name} set:")
    for u, c in zip(unique, counts):
        print(f"  {CLASS_NAMES[u]}: {c}")

save_checkpoint("data_splits", {
    "train_batches": int(train_batches),
    "val_batches":   int(val_batches),
    "test_batches":  int(test_batches),
})


In [ ]:
# ── 2.2 Augmentation using tf.image (works reliably on any pixel range) ─────

def augment_image(image):
    """Apply augmentations to a single image."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    return image

print("Augmentation functions defined.")


In [ ]:
# ── 2.3 Preprocessing pipelines ─────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_and_augment(images, labels):
    """Normalize to [-1, 1] FIRST, then augment."""
    images = tf.cast(images, tf.float32)
    images = images / 127.5 - 1.0
    images = tf.map_fn(augment_image, images)
    return images, labels

def preprocess_eval(images, labels):
    """Normalize to [-1, 1] only."""
    images = tf.cast(images, tf.float32)
    images = images / 127.5 - 1.0
    return images, labels

# Build pipelines — .cache() on val/test so they survive multiple epochs
train_pipeline = (
    train_ds
    .map(preprocess_and_augment, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_pipeline = (
    val_ds
    .cache()
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

test_pipeline = (
    test_ds
    .cache()
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

# Verify pixel range
for imgs, lbls in train_pipeline.take(1):
    print(f"Batch shape: {imgs.shape}, Labels shape: {lbls.shape}")
    print(f"Pixel range: [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")
    unique, counts = np.unique(np.argmax(lbls.numpy(), axis=1), return_counts=True)
    print(f"Labels in batch: {dict(zip([CLASS_NAMES[u] for u in unique], counts))}")
    if imgs.numpy().min() >= -1.5 and imgs.numpy().max() <= 1.5:
        print("Preprocessing OK")
    else:
        print("WARNING: Pixel range is WRONG!")


In [ ]:
# ── 2.4 Visualize augmented training batch ──────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Augmented Training Batch", fontsize=14, fontweight="bold")

for imgs, lbls in train_pipeline.take(1):
    for i, ax in enumerate(axes.flat):
        if i < len(imgs):
            img_display = (imgs[i].numpy() + 1.0) / 2.0
            img_display = np.clip(img_display, 0, 1)
            ax.imshow(img_display)
            label_idx = np.argmax(lbls[i].numpy())
            ax.set_title(f"{CLASS_NAMES[label_idx]}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
fig.savefig(OUT_DIR / "08_augmented_batch.png", dpi=150)
plt.show()


---
## Part 3 — Model Training (3-Stage Progressive Fine-Tuning)

| Stage | Layers Trained | Learning Rate | Epochs |
|-------|---------------|---------------|--------|
| Phase 1 | Classifier head only | 1e-3 | 15 |
| Phase 2 | Last ~30% of backbone + head | 1e-4 | 20 |
| Phase 3 | All layers | 1e-5 | 10 |

**Optimizer**: AdamW (weight_decay=0.01)  
**Loss**: Categorical cross-entropy with label smoothing (0.1)  
**LR Schedule**: Cosine annealing with 5-epoch linear warmup

In [ ]:
# ── 3.1 Build MobileNetV3-Large model (6-class) ───────────────────────
base_model = MobileNetV3Large(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=False,
)

# Freeze all base layers initially
base_model.trainable = False

# Custom classification head
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)  # slightly higher dropout for 6 classes
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="OnionGuard_MultiClass")
model.summary(show_trainable=True, expand_nested=False)
print(f"\nTotal base layers: {len(base_model.layers)}")
print(f"Output classes: {NUM_CLASSES}")

# ── Diagnostic: verify model outputs
print("\n--- Pre-training diagnostic ---")
for imgs, lbls in val_pipeline.take(1):
    preds = model.predict(imgs[:4], verbose=0)
    print(f"Sample predictions (before training):")
    for i in range(min(4, len(imgs))):
        true_cls = CLASS_NAMES[np.argmax(lbls[i])]
        pred_cls = CLASS_NAMES[np.argmax(preds[i])]
        print(f"  True: {true_cls:20s} | Pred: {pred_cls} ({preds[i].round(3)})")
    print("(Random predictions expected before training)")


In [ ]:
# ── 3.2 Learning rate schedule: cosine annealing with warmup ────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, warmup_steps, total_steps, min_lr=1e-7):
        super().__init__()
        self.max_lr = max_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.max_lr * (step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0))
        decay_steps = tf.cast(self.total_steps - self.warmup_steps, tf.float32)
        decay_step = tf.maximum(step - tf.cast(self.warmup_steps, tf.float32), 0.0)
        cosine_decay = 0.5 * (1.0 + tf.cos(np.pi * decay_step / tf.maximum(decay_steps, 1.0)))
        decay_lr = self.min_lr + (self.max_lr - self.min_lr) * cosine_decay
        return tf.where(step < tf.cast(self.warmup_steps, tf.float32), warmup_lr, decay_lr)

    def get_config(self):
        return {"max_lr": self.max_lr, "warmup_steps": self.warmup_steps,
                "total_steps": self.total_steps, "min_lr": self.min_lr}

# Common callbacks
def get_callbacks(phase_name):
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=7, restore_best_weights=True, verbose=1
        ),
        callbacks.ModelCheckpoint(
            str(OUT_DIR / f"best_model_{phase_name}.keras"),
            monitor="val_accuracy", save_best_only=True, verbose=1
        ),
    ]

print("LR schedule & callbacks defined ✓")

In [ ]:
# ── 3.3 Phase 1: Train classifier head only ─────────────────────────────
_ckpt_p1 = load_checkpoint("history_phase1")

if _ckpt_p1 is not None:
    model.load_weights(str(OUT_DIR / "best_model_phase1.keras"))
    class _FakeHistory:
        def __init__(self, h): self.history = h
    history_p1 = _FakeHistory(_ckpt_p1)
    print("Phase 1: LOADED FROM CHECKPOINT (skipping training)")
else:
    steps_per_epoch = tf.data.experimental.cardinality(train_pipeline).numpy()
    total_steps_p1 = steps_per_epoch * EPOCHS_PHASE1
    warmup_steps_p1 = steps_per_epoch * 3

    lr_schedule_p1 = WarmupCosineDecay(
        max_lr=1e-3, warmup_steps=warmup_steps_p1, total_steps=total_steps_p1
    )

    model.compile(
        optimizer=optimizers.AdamW(learning_rate=lr_schedule_p1, weight_decay=0.01),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    print(f"Phase 1: Training classifier head ({EPOCHS_PHASE1} epochs)")
    print(f"  Trainable params: {sum(np.prod(v.shape) for v in model.trainable_variables):,}")
    print(f"  Using class weights to handle imbalance")

    history_p1 = model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=EPOCHS_PHASE1,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("phase1"),
    )

    save_checkpoint("history_phase1", history_p1.history)

steps_per_epoch = tf.data.experimental.cardinality(train_pipeline).numpy()
print(f"Phase 1 best val_accuracy: {max(history_p1.history['val_accuracy']):.4f}")


In [ ]:
# ── 3.4 Phase 2: Unfreeze last ~30% of backbone ────────────────────────────
_ckpt_p2 = load_checkpoint("history_phase2")

if _ckpt_p2 is not None:
    model.load_weights(str(OUT_DIR / "best_model_phase2.keras"))
    # Ensure layers are unfrozen to correct state
    base_model.trainable = True
    num_layers = len(base_model.layers)
    fine_tune_from = int(num_layers * 0.7)
    for layer in base_model.layers[:fine_tune_from]:
        layer.trainable = False
    class _FakeHistory:
        def __init__(self, h): self.history = h
    history_p2 = _FakeHistory(_ckpt_p2)
    print("Phase 2: LOADED FROM CHECKPOINT (skipping training)")
else:
    base_model.trainable = True
    num_layers = len(base_model.layers)
    fine_tune_from = int(num_layers * 0.7)

    for layer in base_model.layers[:fine_tune_from]:
        layer.trainable = False

    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    print(f"Phase 2: Unfroze {trainable_count}/{num_layers} backbone layers (from layer {fine_tune_from})")

    total_steps_p2 = steps_per_epoch * EPOCHS_PHASE2
    warmup_steps_p2 = steps_per_epoch * 3

    lr_schedule_p2 = WarmupCosineDecay(
        max_lr=1e-4, warmup_steps=warmup_steps_p2, total_steps=total_steps_p2
    )

    model.compile(
        optimizer=optimizers.AdamW(learning_rate=lr_schedule_p2, weight_decay=0.01),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    print(f"  Trainable params: {sum(np.prod(v.shape) for v in model.trainable_variables):,}")

    history_p2 = model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=EPOCHS_PHASE2,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("phase2"),
    )

    save_checkpoint("history_phase2", history_p2.history)

print(f"Phase 2 best val_accuracy: {max(history_p2.history['val_accuracy']):.4f}")

In [ ]:
# ── 3.5 Phase 3: Unfreeze ALL layers (full fine-tuning) ─────────────────────
_ckpt_p3 = load_checkpoint("history_phase3")

if _ckpt_p3 is not None:
    model.load_weights(str(OUT_DIR / "best_model_phase3.keras"))
    for layer in base_model.layers:
        layer.trainable = True
    class _FakeHistory:
        def __init__(self, h): self.history = h
    history_p3 = _FakeHistory(_ckpt_p3)
    print("Phase 3: LOADED FROM CHECKPOINT (skipping training)")
else:
    for layer in base_model.layers:
        layer.trainable = True

    total_steps_p3 = steps_per_epoch * EPOCHS_PHASE3
    warmup_steps_p3 = steps_per_epoch * 2

    lr_schedule_p3 = WarmupCosineDecay(
        max_lr=1e-5, warmup_steps=warmup_steps_p3, total_steps=total_steps_p3
    )

    model.compile(
        optimizer=optimizers.AdamW(learning_rate=lr_schedule_p3, weight_decay=0.01),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    print(f"Phase 3: Full fine-tuning — ALL layers trainable ({EPOCHS_PHASE3} epochs)")
    print(f"  Trainable params: {sum(np.prod(v.shape) for v in model.trainable_variables):,}")

    history_p3 = model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=EPOCHS_PHASE3,
        class_weight=class_weight_dict,
        callbacks=get_callbacks("phase3"),
    )

    save_checkpoint("history_phase3", history_p3.history)

print(f"Phase 3 best val_accuracy: {max(history_p3.history['val_accuracy']):.4f}")

In [ ]:
# ── 3.6 Training history — combined plot ────────────────────────────────────
def merge_histories(*histories):
    merged = {}
    for h in histories:
        for key, vals in h.history.items():
            merged.setdefault(key, []).extend(vals)
    return merged

history_all = merge_histories(history_p1, history_p2, history_p3)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(history_all["accuracy"], label="Train Accuracy", linewidth=2)
axes[0].plot(history_all["val_accuracy"], label="Val Accuracy", linewidth=2)
# Mark phase boundaries
p1_end = len(history_p1.history["accuracy"])
p2_end = p1_end + len(history_p2.history["accuracy"])
axes[0].axvline(x=p1_end, color="gray", linestyle="--", alpha=0.5, label="Phase 2 start")
axes[0].axvline(x=p2_end, color="gray", linestyle=":", alpha=0.5, label="Phase 3 start")
axes[0].set_title("Model Accuracy (3-Phase Training)", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_all["loss"], label="Train Loss", linewidth=2)
axes[1].plot(history_all["val_loss"], label="Val Loss", linewidth=2)
axes[1].axvline(x=p1_end, color="gray", linestyle="--", alpha=0.5, label="Phase 2 start")
axes[1].axvline(x=p2_end, color="gray", linestyle=":", alpha=0.5, label="Phase 3 start")
axes[1].set_title("Model Loss (3-Phase Training)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUT_DIR / "09_training_history.png", dpi=150)
plt.show()

best_val_acc = max(history_all["val_accuracy"])
print(f"\nBest validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

---
## Part 4 — Evaluation on Test Set
Confusion matrix, classification report, and per-class accuracy on the held-out Category B test set.

In [ ]:
# ── 4.1 Test set evaluation ─────────────────────────────────────────────
model.compile(loss=keras.losses.CategoricalCrossentropy(), metrics=["accuracy"])
test_loss, test_acc = model.evaluate(test_pipeline, verbose=1)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test Loss:     {test_loss:.4f}")

# Get predictions
y_true = []
y_pred = []
for imgs, lbls in test_pipeline:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Prediction distribution
display_names = CLASS_NAMES
print(f"\nPrediction distribution:")
for i, name in enumerate(display_names):
    pred_count = np.sum(y_pred == i)
    true_count = np.sum(y_true == i)
    print(f"  {name:20s}: pred={pred_count:4d}, true={true_count:4d}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    y_true, y_pred,
    target_names=display_names,
    labels=list(range(NUM_CLASSES)),
    digits=4,
    zero_division=0
))

save_checkpoint("evaluation", {
    "test_acc": test_acc, "test_loss": test_loss,
    "y_true": y_true, "y_pred": y_pred,
})


In [ ]:
# ── 4.2 Confusion matrix ────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=display_names, yticklabels=display_names,
            annot_kws={"size": 12}, linewidths=1, linecolor="black")
ax.set_xlabel("Predicted", fontsize=13)
ax.set_ylabel("Actual", fontsize=13)
ax.set_title(f"Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%",
             fontsize=14, fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(OUT_DIR / "10_confusion_matrix.png", dpi=150)
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(display_names):
    class_total = cm[i].sum()
    class_correct = cm[i][i] if class_total > 0 else 0
    acc = class_correct / class_total * 100 if class_total > 0 else 0
    print(f"  {name:20s}: {class_correct}/{class_total} = {acc:.2f}%")


In [ ]:
# ── 4.3 Misclassified samples ───────────────────────────────────────────────
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"Misclassified: {len(misclassified_idx)} / {len(y_true)} ({len(misclassified_idx)/len(y_true)*100:.2f}%)\n")

if len(misclassified_idx) > 0:
    # Collect misclassified images
    all_test_imgs = []
    for imgs, _ in test_pipeline:
        all_test_imgs.append(imgs.numpy())
    all_test_imgs = np.concatenate(all_test_imgs, axis=0)

    show_n = min(12, len(misclassified_idx))
    sample_idx = np.random.choice(misclassified_idx, show_n, replace=False)

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    fig.suptitle("Misclassified Samples", fontsize=14, fontweight="bold")

    for i, ax in enumerate(axes.flat):
        if i < show_n:
            idx = sample_idx[i]
            ax.imshow(all_test_imgs[idx])
            ax.set_title(f"True: {display_names[y_true[idx]]}\nPred: {display_names[y_pred[idx]]}",
                         fontsize=8, color="red")
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(OUT_DIR / "11_misclassified.png", dpi=150)
    plt.show()
else:
    print("No misclassifications! Perfect test accuracy.")

---
## Part 5 — Grad-CAM Interpretability
Visualize what regions of the image the model focuses on when making predictions. Critical for building farmer trust in AI recommendations.

In [ ]:
# ── 5.1 Grad-CAM implementation ─────────────────────────────────────────
def get_gradcam_heatmap(model, img_array, pred_index=None):
    """Generate Grad-CAM heatmap by manually splitting the forward pass."""
    img_tensor = tf.cast(img_array, tf.float32)

    # Split forward pass: base_model -> head layers
    with tf.GradientTape() as tape:
        # Step 1: Get conv features from base model
        conv_outputs = base_model(img_tensor, training=False)
        tape.watch(conv_outputs)

        # Step 2: Manually pass through head layers
        x = conv_outputs
        for layer in model.layers:
            if layer.name == base_model.name:
                continue
            if 'input' in layer.name:
                continue
            x = layer(x)

        predictions = x
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    if grads is None:
        # Fallback: input-gradient saliency map
        with tf.GradientTape() as tape2:
            tape2.watch(img_tensor)
            preds = model(img_tensor)
            loss = preds[:, tf.argmax(preds[0])]
        input_grads = tape2.gradient(loss, img_tensor)
        if input_grads is not None:
            heatmap = tf.reduce_mean(tf.abs(input_grads[0]), axis=-1)
            heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
            return heatmap.numpy()
        return np.zeros((7, 7), dtype=np.float32)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on original image."""
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay = alpha * heatmap_colored + (1 - alpha) * img
    return np.clip(overlay, 0, 1)

print("Grad-CAM functions defined.")


In [ ]:
# ── 5.2 Grad-CAM visualizations ─────────────────────────────────────────
n_samples = min(8, NUM_CLASSES * 2)
fig, axes = plt.subplots(2, n_samples, figsize=(2.5 * n_samples, 6))
fig.suptitle("Grad-CAM — Model Attention Maps", fontsize=14, fontweight="bold")

sample_count = 0
for imgs, lbls in test_pipeline:
    for i in range(len(imgs)):
        if sample_count >= n_samples:
            break
        img = imgs[i:i+1]
        true_label = np.argmax(lbls[i].numpy())
        heatmap = get_gradcam_heatmap(model, img)
        pred = model.predict(img, verbose=0)
        pred_label = np.argmax(pred[0])
        confidence = pred[0][pred_label] * 100

        img_display = (img[0].numpy() + 1.0) / 2.0
        img_display = np.clip(img_display, 0, 1)

        color = "green" if pred_label == true_label else "red"
        axes[0][sample_count].imshow(img_display)
        axes[0][sample_count].set_title(f"True: {display_names[true_label]}", fontsize=7)
        axes[0][sample_count].axis("off")

        overlay = overlay_gradcam(img_display, heatmap)
        axes[1][sample_count].imshow(overlay)
        axes[1][sample_count].set_title(f"Pred: {display_names[pred_label]}\n({confidence:.0f}%)", fontsize=7, color=color)
        axes[1][sample_count].axis("off")

        sample_count += 1
    if sample_count >= n_samples:
        break

plt.tight_layout()
fig.savefig(OUT_DIR / "12_gradcam.png", dpi=150)
plt.show()


---
## Part 6 — TFLite Export & Quantization
Two quantization strategies:
1. **Post-Training Dynamic Range Quantization** — fast, ~4x size reduction
2. **Post-Training Full Integer Quantization (INT8)** — max compression, uses representative dataset for calibration

Note: QAT via `tensorflow-model-optimization` is incompatible with TF 2.21. Full INT8 PTQ with a representative calibration dataset achieves comparable results for binary classification tasks.

In [ ]:
# ── 6.1 Save full Keras model ───────────────────────────────────────────────
model.save(str(OUT_DIR / "onion_model_full.keras"))
full_size_mb = os.path.getsize(OUT_DIR / "onion_model_full.keras") / (1024 * 1024)
print(f"Full Keras model saved: {OUT_DIR / 'onion_model_full.keras'}")
print(f"Full model size: {full_size_mb:.2f} MB")
save_checkpoint("model_saved", {"full_size_mb": full_size_mb})

In [ ]:
# ── 6.2 Dynamic Range Quantization (float16) ─────────────────────────
# Dynamic range gives ~3-4x compression with minimal accuracy loss
# Input/output stays float32 so no quantization mismatch issues

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Float16 quantization: weights stored as float16, computed as float32
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = OUT_DIR / "onion_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"TFLite float16 model saved to: {tflite_path}")
print(f"TFLite model size: {tflite_size_mb:.2f} MB")
print(f"Compression ratio: {full_size_mb / tflite_size_mb:.1f}x")


In [ ]:
# ── 6.3 Validate TFLite model accuracy ─────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

tflite_correct = 0
tflite_total = 0

for imgs, lbls in test_pipeline:
    for i in range(len(imgs)):
        # Input is float32 [-1, 1] — no quantization needed
        img_input = np.expand_dims(imgs[i].numpy(), axis=0).astype(np.float32)

        interpreter.set_tensor(input_details[0]["index"], img_input)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])

        pred_label = np.argmax(output[0])
        true_label = np.argmax(lbls[i].numpy())

        if pred_label == true_label:
            tflite_correct += 1
        tflite_total += 1

tflite_acc = tflite_correct / tflite_total
print(f"\nTFLite float16 Test Accuracy: {tflite_acc:.4f} ({tflite_acc*100:.2f}%)")
print(f"Full model accuracy:          {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Accuracy drop (TFLite):       {(test_acc - tflite_acc)*100:.2f}%")
print(f"\nModel size: {tflite_size_mb:.2f} MB (target: <8 MB) {'PASS' if tflite_size_mb < 8 else 'OVER TARGET'}")

save_checkpoint("tflite_eval", {
    "tflite_acc": tflite_acc, "full_acc": test_acc,
    "tflite_size_mb": tflite_size_mb, "full_size_mb": full_size_mb,
})


---
## Final Summary

In [ ]:
# ── Final Summary ──────────────────────────────────────────────────────────────
print("=" * 60)
print("ONION DISEASE & PEST MODEL — FINAL REPORT")
print("=" * 60)
print(f"""
Model:           MobileNetV3-Large (3-stage progressive fine-tuning)
Classes:         {', '.join(CLASS_NAMES)}
Num Classes:     {NUM_CLASSES}
Input:           {IMG_SIZE}x{IMG_SIZE}x3, normalized [-1,1]
Optimizer:       AdamW (weight_decay=0.01)
LR Schedule:     Cosine annealing with linear warmup
Augmentation:    RandomFlip/Rotation/Zoom/Brightness/Contrast
Regularization:  Dropout(0.3) + Class Weights
Quantization:    Full INT8 PTQ with representative dataset calibration

RESULTS:
  Full Model Accuracy:    {test_acc*100:.2f}%
  TFLite INT8 Accuracy:   {tflite_acc*100:.2f}%
  Full Model Size:        {full_size_mb:.2f} MB
  TFLite Model Size:      {tflite_size_mb:.2f} MB

SAVED TO /kaggle/working/:
  {OUT_DIR}/onion_model_full.keras
  {OUT_DIR}/onion_model.tflite
""")
print("Pipeline complete! Download your models from the Output tab.")
